# Entity resolution, step by step

This notebook runs the pipeline that `scripts/resolve.py` runs on a file of any size: the same `er.run.Run`, one step
per section, on a small generated population (about 3,700 records) so that every step can be seen. It holds no logic
of its own; each section calls the code, then charts what the step did. Every code cell prints its Big-O, filled in
with this file's numbers, and its time. The whole notebook runs in about a minute.

The run never reads the truth (which records are one entity). It exists only because the data is generated, and the
notebook reads it to show what each step got right.

1 Setup · 2 Generate · 3 Standardize · 4 Rules · 5 Candidates · 6 Fit · 7 Score · 8 Why a pair matches · 9 Threshold ·
10 Cluster and evaluate · 11 Increment · 12 How the configuration was chosen · 13 Cost as the file grows

N records, L characters per value, P pairs inside a key's blocks, C candidate pairs, K comparisons, S sampled pairs.

## 1 · Setup

The chosen pipeline is `config/final.toml`, which `scripts/choose.py` wrote. The demo's population is
`config/generator.toml`'s, made smaller. Charts use one colour per meaning: blue for the same entity (or evidence for
a match), orange for different entities, red for evidence against.

In [ ]:
import logging
import math
import shutil
import time
from contextlib import contextmanager
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, PercentFormatter
from matplotlib_inline.backend_inline import set_matplotlib_formats

from er.candidates import found_by
from er.config import column_name, load_pipeline
from er.evaluate import cluster_metrics, missed_pairs, pair_metrics, predicted_pairs
from er.generate import check, generate, load_reference, load_spec
from er.io import table_after
from er.match import SCORED_U, parameters, scaled_prior, waterfall
from er.resolve import connected_components
from er.run import Run
from er.settings import load_settings

logging.disable(logging.WARNING)
S = load_settings()
FINAL = load_pipeline(S.root / "config" / "final.toml", S.standardize, S.chains, S.columns)
SPEC = replace(load_spec(S.root / "config" / "generator.toml")[0], entities=3_000)  # the demo's size
DEMO = S.paths["artifacts"] / "demo"
DEMO.mkdir(parents=True, exist_ok=True)
UID, ENT = S.data.unique_id, S.data.entity_id
L, R = f"{UID}_l", f"{UID}_r"
FIELDS = [f for f in S.columns if f not in ("id", "entity")]  # the logical fields of config/settings.toml [columns]
pd.set_option("display.max_columns", 30, "display.width", 200)

BLUE, ORANGE, RED, INK, INK2, MUTED, GRID, AXIS, SURFACE = ("#2a78d6", "#eb6834", "#e34948", "#0b0b0b", "#52514e",
                                                           "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb")
set_matplotlib_formats("svg")
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2, "axes.titlecolor": INK,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
                     "grid.color": GRID, "grid.linewidth": 0.6, "xtick.color": MUTED, "ytick.color": MUTED,
                     "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "font.size": 9, "lines.linewidth": 2,
                     "legend.frameon": False, "figure.dpi": 100})


@contextmanager
def timed(big_o: str):  # a cell's own work, outside the run's steps
    t = time.perf_counter()
    yield
    print(f"{big_o} · {time.perf_counter() - t:.2f}s")


def done(run: Run) -> None:  # the step just run: its Big-O with this file's numbers, and its time
    s = run.steps[-1]
    print(f"{s['step']}: {s['big_o']} = {s['filled']} · {s['seconds']:.2f}s")


def same(pairs: pd.DataFrame) -> np.ndarray:  # the truth, read only to show what a step got right
    return pairs[L].map(entity).to_numpy() == pairs[R].map(entity).to_numpy()


def finish(fig) -> None:
    fig.tight_layout()
    plt.show()

## 2 · Generate

A synthetic population built from public reference data: each entity, a person or a company, is written as one or
more records with typos, empty fields and life changes (a move, a new phone, a married surname). The truth is kept
apart. Below: the entity whose three records differ the most, then how many records each entity has and how often
each column is filled. O(E·r) for E entities of r records each.

In [ ]:
with timed(f"generate: O(E·r) = O({SPEC.entities:,} entities · their records)"):
    g = generate(SPEC, *load_reference(S.paths["reference"]))
    check(SPEC, g)
g.df.to_parquet(DEMO / "records.parquet", index=False)
entity = g.truth.set_index(UID)[ENT]
per_entity = entity.value_counts()
print(f"{len(g.df):,} records · {len(per_entity):,} entities · {(per_entity > 1).mean():.0%} with more than one record")

raw = g.df.set_index(UID)
varied = raw.groupby(entity.reindex(raw.index).to_numpy()).nunique().gt(1).sum(axis=1)  # columns that differ, per entity
example = varied[per_entity.reindex(varied.index).eq(3)].idxmax()
ex_ids = entity.index[entity == example]
display(raw.loc[ex_ids].dropna(axis=1, how="all"))

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8), width_ratios=[1, 2])
counts = per_entity.value_counts().sort_index()
a.bar(counts.index.astype(str), counts.to_numpy(), width=0.6, color=BLUE)
a.set(title="Records per entity", xlabel="records", ylabel="entities")
a.grid(axis="x", visible=False)
fill = raw.notna().mean().sort_values()
b.barh(fill.index, fill.to_numpy(), height=0.6, color=BLUE)
b.set(title="Records with the column filled", xlim=(0, 1))
b.grid(axis="y", visible=False)
b.xaxis.set_major_formatter(PercentFormatter(1))
b.tick_params(axis="y", labelsize=7)
finish(fig)

## 3 · Standardize

`load` reads the file in chunks and checks it against `config/pipeline.toml` `[fields]`: the id is mandatory, and at
least one of the person and company names is needed. An optional field the file lacks disables only what reads it.
It then builds each logical field from `config/settings.toml` `[columns]` (the first filled source wins) and the
normalizations the pipeline reads: kana folding, legal forms dropped, addresses cut to the street number. Each record
is standardized on its own, so chunks change nothing. O(N·L).

In [ ]:
run = Run(FINAL, S)
run.load(DEMO / "records.parquet")
done(run)
print("disabled:", ", ".join(run.disabled) or "nothing: every optional field is in the file")
std = run.con.sql("SELECT * FROM records").df().set_index(UID)

display(std.loc[ex_ids, FIELDS])  # the entity above, in logical fields
rows = []
for field, chain in [("person_name", "fold"), ("company_name", "company"), ("address", "address")]:
    after = column_name(field, S.chains[chain])
    changed = std.loc[std[field].notna() & (std[field] != std[after]), [field, after]].head(3)
    rows.append(changed.set_axis(["before", "after"], axis=1).assign(field=field, normalization=" → ".join(S.chains[chain])))
display(pd.concat(rows)[["field", "normalization", "before", "after"]])

## 4 · Rules

Stage 1 is deterministic. An exact rule joins records equal on all its keys; a fuzzy rule needs some keys equal and
others close (names by Jaro–Winkler, a phone one digit off). Before any join, `budget` counts every rule's pairs
P = Σ C(k, 2) over its key blocks and stops the run if one is over budget. The first rule in order wins a pair, and a
fuzzy pair is kept only if it touches a record the exact rules left alone. Left: how many true pairs the rules join,
clusters included. Right: the true pairs each rule joins itself.

In [ ]:
plan = run.budget()
done(run)
run.rules()
done(run)
kept = run.con.sql("SELECT * FROM rule_kept").df().assign(true=same)
order = [r.name for st in FINAL.rules.stages for r in st.rules]
stage_of = {r.name: st.name for st in FINAL.rules.stages for r in st.rules}
per_rule = (plan[plan["kind"] == "rule"].set_index("name")[["est_pairs", "largest_block"]].rename(columns={"est_pairs": "P"})
            .join(kept.groupby("rule")["true"].agg(joined="size", true="sum")).fillna(0).astype(int).reindex(order))
display(per_rule.rename_axis("rule"))

rule_clusters = run.con.sql("SELECT * FROM rule_clusters").df().set_index(UID)["cluster_id"]
exact_only = connected_components(pd.Series(std.index), kept[kept["stage"] == "exact"], run.db, UID)
m_exact, m_rules = (pair_metrics(c, entity.reindex(c.index)) for c in (exact_only, rule_clusters))
total = m_rules["TP"] + m_rules["FN"]
print(f"the rules join {m_rules['recall']:.1%} of the {total:,} true pairs, with {m_rules['FP']} false")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4), width_ratios=[1, 1.3])
funnel = pd.Series({"true pairs": total, "joined by the exact rules": m_exact["TP"], "… and the fuzzy rules": m_rules["TP"]})[::-1]
a.barh(funnel.index, funnel.to_numpy(), height=0.6, color=BLUE)
for y, v in enumerate(funnel.to_numpy()):
    a.text(v, y, f"  {v / total:.0%}", va="center", color=INK2)
a.set(title="True pairs joined by the rules", xlim=(0, total * 1.15))
a.grid(axis="y", visible=False)
y = np.arange(len(order))[::-1]
for stage, colour in (("exact", BLUE), ("fuzzy", ORANGE)):
    mask = np.array([stage_of[r] == stage for r in order])
    b.barh(y[mask], per_rule["true"].to_numpy()[mask], height=0.6, color=colour, label=f"{stage} rules")
b.set_yticks(y, order, fontsize=7)
b.set(title="True pairs each rule joins (the first rule wins a pair)")
b.grid(axis="y", visible=False)
b.legend(loc="lower right")
finish(fig)

## 5 · Candidates

Stage 2 compares only pairs that some generator lists. A key generator pairs records equal on all its keys; a
deletion-key generator pairs names one edit apart (each value and its one-character deletions, within a postal
area); a sorted window pairs neighbours in sorted order. Each is one SQL join or window over the records table; their
pairs are merged, and pairs inside one rule cluster are dropped. Every key here pairs a share of the records that stays
flat as the file grows: a field with a fixed pool of values (the phone's last digits, an address shared by a household)
is always combined with another. O(N·L + Σ P_g).

In [ ]:
report = run.candidates()
done(run)
cands = run.con.sql("SELECT * FROM cands").df().assign(true=same)
targets = missed_pairs(rule_clusters, entity.reindex(rule_clusters.index), UID)  # true pairs the rules left apart
wanted = set(zip(targets[L].astype(str), targets[R].astype(str)))
on_target = np.array([k in wanted for k in zip(cands[L], cands[R])])
report["finds"] = [(found_by(cands, {name}) & on_target).sum() / len(wanted) for name in report["generator"]]
report["different_entity_pairs"] = [(found_by(cands, {name}) & ~cands["true"]).sum() for name in report["generator"]]
display(report[["generator", "big_o", "pairs", "finds", "different_entity_pairs", "seconds"]].round(3))
print(f"{len(cands):,} candidate pairs · together they list {on_target.sum() / len(wanted):.1%} of the {len(wanted):,} true pairs the rules left apart")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
y = np.arange(len(report))[::-1]
a.barh(y, report["pairs"], height=0.6, color=BLUE)
a.set(title="Pairs each generator lists", xscale="log")
b.barh(y, report["finds"], height=0.6, color=BLUE)
b.set(title="Share of the true pairs the rules left apart it finds", xlim=(0, 1))
b.xaxis.set_major_formatter(PercentFormatter(1))
a.set_yticks(y, report["generator"], fontsize=7)
for ax in (a, b):
    ax.grid(axis="y", visible=False)
finish(fig)

## 6 · Fit, without labels

Fellegi–Sunter weighs each comparison's agreement level by how much likelier it is for a true match than by chance:
log2(m/u) bits. With no labels, λ (the chance two random records match) comes from near-certain rules and their
measured recall; u (agreement by chance) from value counts for the exact levels and a random sample of pairs for the
rest; m (agreement for true matches) by EM on blocks of equal phones or addresses. The fit uses a fixed-size sample of
whole postal areas, so its cost doesn't grow with the file. O(n + S·K + P_em·K).

In [ ]:
model = run.fit()
done(run)
lam = scaled_prior(model, run.n)
print(f"λ = {lam:.2e} at {run.n:,} records: a pair starts at {math.log2(lam / (1 - lam)):.1f} bits")
params = parameters(model).dropna(subset=["weight"]).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.barh(params["comparison"] + " · " + params["level"], params["weight"], height=0.6,
        color=np.where(params["weight"] >= 0, BLUE, RED))
ax.axvline(0, color=AXIS, linewidth=0.8)
ax.set(title="Bits of evidence per comparison level, log2(m/u)", xlabel="bits")
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=7)
finish(fig)

## 7 · Score

u is measured again on the file being scored: how often two records agree by chance depends on how many records
there are, and u from a smaller file would overstate a rare value's evidence. Then every candidate pair gets its
weight: the prior plus each comparison's bits. A pair is accepted at the threshold `scripts/choose.py` chose.
O(N + S·K) + O(C·K·L²).

In [ ]:
run.score()
done(run)
scores = run.con.sql("SELECT * FROM scores").df().assign(same=same)
threshold = run.pipeline.fs.min_match_weight
w = scores["match_weight"]
print(f"{len(scores):,} pairs scored · {(w >= threshold).sum():,} reach {threshold:g} bits, "
      f"{(scores['same'] & (w >= threshold)).sum():,} of them true")

fig, ax = plt.subplots(figsize=(10, 3.4))
bins = np.arange(math.floor(w.min()), math.ceil(w.max()) + 1)
ax.hist(w[scores["same"]], bins, histtype="step", color=BLUE, linewidth=2, label="same entity")
ax.hist(w[~scores["same"]], bins, histtype="step", color=ORANGE, linewidth=2, label="different entities")
ax.axvline(threshold, color=INK2, linewidth=1)
ax.text(threshold, ax.get_ylim()[1], f" threshold {threshold:g} bits", va="top", color=INK2)
ax.set(title="Weights of the candidate pairs", xlabel="match weight (bits)", ylabel="pairs", yscale="log")
ax.legend(loc="upper right")
finish(fig)

## 8 · Why a pair matches

A weight is a sum, so it can be read term by term. Two pairs: the accepted true match closest to the threshold whose
names differ, and the rejected pair of different entities closest to it. Each bar is one term: the prior, then the
bits of the level each comparison landed in.

In [ ]:
show = list(FINAL.explain_show)
filled = list(FINAL.explain_filled)
left, right = std.loc[scores[L], show].set_index(scores.index), std.loc[scores[R], show].set_index(scores.index)
both_filled = (left[filled].notna() & right[filled].notna()).all(axis=1)
differs = (left[filled[0]] != right[filled[0]]).fillna(False)
accepted = w >= threshold
picked = {"Accepted: a true match with a typo": scores[accepted & scores["same"] & both_filled & differs].nsmallest(1, "match_weight"),
          "Rejected: different entities": scores[~accepted & ~scores["same"]].nlargest(1, "match_weight")}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
ends = []
for ax, (title, pair) in zip(axes, picked.items()):
    row = pair.iloc[0]
    display(std.loc[[row[L], row[R]], show])
    bits = waterfall(run.scoring, {c.name: row[f"gamma_{c.name}"] for c in run.pipeline.fs.model.comparisons}, run.n)
    end = bits["bits"].cumsum().to_numpy()
    start = np.concatenate([[0], end[:-1]])
    ends += [*start, *end]
    ax.bar(bits["evidence"], bits["bits"], bottom=start, width=0.6, color=np.where(bits["bits"] >= 0, BLUE, RED))
    ax.plot(len(bits) - 0.5, end[-1], marker="o", markersize=7, color=INK, clip_on=False)
    ax.axhline(threshold, color=INK2, linewidth=1)
    ax.text(-0.4, threshold, f"threshold {threshold:g}", va="bottom", color=INK2)
    ax.axhline(0, color=AXIS, linewidth=0.8)
    ax.set(title=f"{title}: {row['match_weight']:.1f} bits")
    ax.grid(axis="x", visible=False)
axes[0].set(ylabel="running total (bits)", ylim=(min(ends) - 2, max(ends + [threshold]) + 2))
finish(fig)

## 9 · Threshold

`scripts/choose.py` chose the threshold on cross-validated labelled data: the lowest weight at which no false pair
was accepted and the upper bound of the false-merge rate stays under 5%, then one step stricter. A run without labels
uses it as it is. Here, with this file's truth: what each weight would accept.

In [ ]:
with timed(f"O(C·G) = O({len(scores):,} pairs · 25 weights)"):
    grid = np.arange(-4.0, 21.0)
    curve = pd.DataFrame({"weight": grid,
                          "recall": [(scores["same"] & (w >= x)).sum() / len(wanted) for x in grid],
                          "false_pairs": [(~scores["same"] & (w >= x)).sum() for x in grid]})

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
a.plot(curve["weight"], curve["recall"], color=BLUE)
a.set(title="Share of the true pairs the rules left apart, accepted", xlabel="threshold (bits)", ylim=(0, 1))
a.yaxis.set_major_formatter(PercentFormatter(1))
b.plot(curve["weight"], curve["false_pairs"], color=ORANGE)
b.set(title="Pairs of different entities accepted", xlabel="threshold (bits)")
for ax in (a, b):
    ax.axvline(threshold, color=INK2, linewidth=1)
finish(fig)

## 10 · Cluster and evaluate

The accepted pairs and the rules' pairs are clustered together (connected components, in SQL). Gold gives each
record its cluster and what resolved it: a rule, or the model. Then, with the truth: pairwise recall with its 95%
interval, false merges, and the upper bound of the false-merge rate. Below: cluster sizes against true entity sizes,
and the largest cluster. O(C + (N + E)·d).

In [ ]:
run.cluster()
done(run)
shutil.rmtree(DEMO / "run", ignore_errors=True)
run.write(DEMO / "run")
done(run)
gold = pd.read_parquet(DEMO / "run" / "gold.parquet").set_index(UID)
m = cluster_metrics(gold["cluster_id"], entity.reindex(gold.index))
print(f"recall {m['recall']:.1%} ({m['recall_lo']:.1%}–{m['recall_hi']:.1%}) · false merges {m['FP']} "
      f"(rate at most {m['fp_rate_hi']:.2%}) · precision {m['precision']:.2%}")
display(gold["resolved_by"].fillna("(alone)").value_counts().rename_axis("resolved by").to_frame("records"))

sizes = lambda ids: ids.value_counts().clip(upper=5).value_counts().reindex(range(1, 6), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3))
x = np.arange(1, 6)
ax.bar(x - 0.16, sizes(gold["cluster_id"]), width=0.3, color=BLUE, label="clusters found")
ax.bar(x + 0.16, sizes(entity), width=0.3, color=ORANGE, label="true entities")
ax.set_xticks(x, ["1", "2", "3", "4", "5+"])
ax.set(title="Records per cluster, against records per true entity", xlabel="records", yscale="log")
ax.grid(axis="x", visible=False)
ax.legend()
finish(fig)
largest = gold["cluster_id"].value_counts().index[0]
display(std.loc[gold.index[gold["cluster_id"] == largest], show].assign(entity=entity))

## 11 · Increment

New records only. An increment joins the batch against all the records, rule by rule and generator by generator,
scores the pairs that touch it, and merges into the stored clusters, each existing cluster one node. So its cost grows
with the batch, not with the file. Here: a full run on 90% of the records with the model fitted above, then the other
10% as an increment, compared with the full run on all of them in section 10. At this size the u sample inside
`score` (a fixed S pairs) costs the most in both; the steps that follow the data (rules, candidates, cluster) follow
the batch.

In [ ]:
batch = np.random.default_rng(0).random(len(g.df)) < 0.1
g.df[~batch].to_parquet(DEMO / "first_90.parquet", index=False)
g.df[batch].to_parquet(DEMO / "new_10.parquet", index=False)
shutil.rmtree(DEMO / "increment", ignore_errors=True)
SCORED_U.clear()  # u of exactly these records is cached from section 7; a new batch makes a new set, so measure it again
base = Run(FINAL, S, model=model).full(DEMO / "first_90.parquet", DEMO / "increment")
inc = Run(FINAL, S).increment(DEMO / "new_10.parquet", DEMO / "increment")
for s in inc.steps:
    print(f"increment {s['step']}: {s['big_o']} = {s['filled']} · {s['seconds']:.2f}s")

after = pd.read_parquet(DEMO / "increment" / "gold.parquet").set_index(UID)["cluster_id"]
new_ids = set(g.df.loc[batch, UID])
old_in = after[~after.index.isin(new_ids)].unique()
joined = after[after.index.isin(new_ids) & after.isin(old_in)]
truth = predicted_pairs(entity)
by_inc, by_full = predicted_pairs(after), predicted_pairs(gold["cluster_id"])
print(f"{batch.sum():,} new records · {len(joined):,} joined an existing cluster · pairs only in the increment: "
      f"{len((by_inc - by_full) & truth)} true, {len((by_inc - by_full) - truth)} false · only in the full run: "
      f"{len((by_full - by_inc) & truth)} true, {len((by_full - by_inc) - truth)} false")

steps = ["load", "budget", "fit", "rules", "candidates", "score", "cluster", "write"]
seconds = lambda r: pd.Series({s["step"]: s["seconds"] for s in r.steps}).reindex(steps, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3.2))
y = np.arange(len(steps))[::-1]
ax.barh(y + 0.16, seconds(base), height=0.3, color=BLUE, label=f"full run, {(~batch).sum():,} records (model given)")
ax.barh(y - 0.16, seconds(inc), height=0.3, color=ORANGE, label=f"increment, {batch.sum():,} records")
ax.set_yticks(y, steps)
ax.set(title="Seconds per step", xlabel="seconds", xscale="log")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
finish(fig)

## 12 · How the configuration was chosen

`scripts/choose.py` starts from `config/pipeline.toml` and tries the changes in `config/experiments.py` in order, each
on top of the choice so far, on labelled synthetic data. A change is adopted by the decision rule: "better" needs a
significant paired gain on the selection set; "cheaper" and "safer", no true pair lost; "chosen" is the owner's call,
where no false pair may be added. All need no loss on validation. Test is scored once, at the end. Its tables are in
`reports/choose.md`; read here, not recomputed.

In [ ]:
text = (S.root / "reports" / "choose.md").read_text(encoding="utf-8")
for stage in ("## Stage 1: rules", "## Stage 2: Fellegi–Sunter"):
    decisions = table_after(text.split(stage, 1)[1], "### Decisions")
    print(stage[3:])
    display(decisions.iloc[:, :8])
print(next(line for line in (S.root / "config" / "final.toml").read_text(encoding="utf-8").splitlines() if line.startswith("# Adopted")))

## 13 · Cost as the file grows

Every step's Big-O, for this full run and for the increment, with the numbers filled in. A full run is
O(N·L + Σ P + C·K·L² + (N + E)·d), plus the fit and the u sample, whose sizes are fixed. When every key's pairs
per record stay flat, that is linear in N. An increment of Δ records costs O(|Δ|·(L + Σ b) + C_Δ·K·L²) plus scans of
the stored records. Below, measured by `scripts/benchmark.py` (`reports/benchmark.md`): each file resolved in a
process of its own, up to a million records.

In [ ]:
full_steps = pd.DataFrame(run.steps)[["step", "big_o", "filled", "seconds", "peak_memory_gb"]]
inc_steps = pd.DataFrame(inc.steps)[["step", "big_o", "filled", "seconds"]]
display(full_steps, inc_steps)

curve = table_after((S.root / "reports" / "benchmark.md").read_text(encoding="utf-8"), "## Cost")
curve = curve[pd.to_numeric(curve["seconds"], errors="coerce").notna()].astype({"records": int, "seconds": float, "peak_memory_gb": float})
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, col, title in ((a, "seconds", "Seconds, end to end"), (b, "peak_memory_gb", "Peak memory (GB)")):
    ax.plot(curve["records"], curve[col], color=BLUE, marker="o", markersize=6)
    ax.set(title=title, xlabel="records (log scale)", xscale="log", ylim=(0, curve[col].max() * 1.15))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    last = curve.iloc[-1]
    ax.annotate(f"{last[col]:g} at {last['records']:,}", (last["records"], last[col]), textcoords="offset points",
                xytext=(-8, 8), ha="right", color=INK2)
finish(fig)